In [8]:
import os

base_path = "/Users/jaeeponde/Thesis/Data_100"

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

for subfolder in sorted(os.listdir(base_path)):
    subfolder_path = os.path.join(base_path, subfolder)
    
    if os.path.isdir(subfolder_path):
        count = sum(
            1 for file in os.listdir(subfolder_path)
            if file.lower().endswith(image_extensions)
        )
        
        print(f"{subfolder}: {count} images")

Cars: 221 images
Cats: 202 images
Dogs: 314 images
Rangoli: 358 images
memes: 6991 images
microscopy: 465 images


In [9]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# -------- CONFIG --------
base_path = "/Users/jaeeponde/Thesis/Data_100"
num_images = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- MODEL --------
resnet = models.resnet50(pretrained=True)
resnet = resnet.to(device)
resnet.eval()

# Remove final FC layer → get penultimate (2048-dim vector)
model = nn.Sequential(*list(resnet.children())[:-1])  # output: (B, 2048, 1, 1)

# -------- TRANSFORMS --------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -------- HELPERS --------
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

def load_image(path):
    img = Image.open(path).convert("RGB")
    return transform(img)

# -------- MAIN LOOP --------
for subfolder in sorted(os.listdir(base_path)):
    subfolder_path = os.path.join(base_path, subfolder)
    
    if not os.path.isdir(subfolder_path):
        continue

    # Get first 75 images
    images = [
        os.path.join(subfolder_path, f)
        for f in sorted(os.listdir(subfolder_path))
        if f.lower().endswith(image_extensions)
    ][:num_images]

    if len(images) == 0:
        continue

    activations = []

    for img_path in images:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)

            with torch.no_grad():
                feat = model(img)  # (1, 2048, 1, 1)
                feat = feat.view(-1)  # (2048,)

                # Normalize to [0, 1]
                feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)

                activations.append(feat.cpu().numpy())

        except Exception as e:
            print(f"Error with {img_path}: {e}")

    if len(activations) == 0:
        continue

    activations = np.stack(activations)  # (75, 2048)

    # Average across images → shape (2048,)
    avg_activation = activations.mean(axis=0)

    # -------- PLOT --------
    plt.figure(figsize=(10, 4))
    plt.plot(avg_activation)
    plt.title(f"{subfolder} - Avg Activation (ResNet50 Penultimate Layer)")
    plt.xlabel("Neuron Index (0–2047)")
    plt.ylabel("Avg Normalized Activation")
    plt.grid(alpha=0.3)
    
    # Save instead of show (recommended)
    plt.savefig(f"{subfolder}_activation_plot.png")
    plt.close()

    print(f"Done: {subfolder}")

Done: Cars
Done: Cats
Done: Dogs
Done: Rangoli
Done: memes
Done: microscopy


In [11]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy

# -------- CONFIG --------
base_path = "/Users/jaeeponde/Thesis/Data_100"
num_images_avg = 100
num_images_test = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

# -------- MODEL --------
resnet = models.resnet50(pretrained=True).to(device)
resnet.eval()

# penultimate layer (2048-d)
model = nn.Sequential(*list(resnet.children())[:-1])

# -------- TRANSFORM --------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image(path):
    return transform(Image.open(path).convert("RGB"))

# 🔥 IMPORTANT: no probability normalization here
def get_activation(img_tensor):
    with torch.no_grad():
        feat = model(img_tensor).view(-1)

        # min-max normalize ONLY
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)

        return feat.cpu().numpy()

# -------- MAIN --------
results = {}

for subfolder in sorted(os.listdir(base_path)):
    subfolder_path = os.path.join(base_path, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    images = [
        os.path.join(subfolder_path, f)
        for f in sorted(os.listdir(subfolder_path))
        if f.lower().endswith(image_extensions)
    ]

    if len(images) < (num_images_avg + num_images_test):
        continue

    # ---------- STEP 1: AGGREGATE (first 100 images) ----------
    avg_activations = []

    for img_path in images[:num_images_avg]:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)
            feat = get_activation(img)
            avg_activations.append(feat)
        except:
            continue

    avg_activations = np.stack(avg_activations)

    # 🔥 aggregate first
    avg_dist = avg_activations.mean(axis=0)

    # 🔥 normalize ONCE here
    avg_dist = avg_dist / (avg_dist.sum() + 1e-8)

    # ---------- STEP 2: KL for last 25 ----------
    kl_values = []

    for img_path in images[-num_images_test:]:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)
            feat = get_activation(img)

            # 🔥 normalize ONLY here for KL
            feat = feat / (feat.sum() + 1e-8)

            kl = entropy(feat + 1e-8, avg_dist + 1e-8)
            kl_values.append(kl)

        except:
            continue

    if len(kl_values) == 0:
        continue

    avg_kl = np.mean(kl_values)
    results[subfolder] = avg_kl

    print(f"{subfolder}: Avg KL = {avg_kl:.6f}")

# -------- FINAL OUTPUT --------
print("\n=== FINAL RESULTS (ResNet50 Improved) ===")
for k, v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{k}: {v:.6f}")

Cars: Avg KL = 0.199026
Cats: Avg KL = 0.193674
Dogs: Avg KL = 0.250367
Rangoli: Avg KL = 0.383853
memes: Avg KL = 0.263350
microscopy: Avg KL = 0.357604

=== FINAL RESULTS (ResNet50 Improved) ===
Rangoli: 0.383853
microscopy: 0.357604
memes: 0.263350
Dogs: 0.250367
Cars: 0.199026
Cats: 0.193674


In [13]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy

# -------- CONFIG --------
base_path = "/Users/jaeeponde/Thesis/Data_100"
num_images_avg = 100
num_images_test = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

# -------- MODEL --------
model_full = models.mobilenet_v2(pretrained=True).to(device)
model_full.eval()

# penultimate layer (~1280-d)
model = nn.Sequential(
    model_full.features,
    nn.AdaptiveAvgPool2d((1, 1))
)

# -------- TRANSFORM --------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image(path):
    return transform(Image.open(path).convert("RGB"))

# 🔥 IMPORTANT: no probability normalization here
def get_activation(img_tensor):
    with torch.no_grad():
        feat = model(img_tensor).view(-1)

        # min-max normalize ONLY
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)

        return feat.cpu().numpy()

# -------- MAIN --------
results = {}

for subfolder in sorted(os.listdir(base_path)):
    subfolder_path = os.path.join(base_path, subfolder)
    if not os.path.isdir(subfolder_path):
        continue

    images = [
        os.path.join(subfolder_path, f)
        for f in sorted(os.listdir(subfolder_path))
        if f.lower().endswith(image_extensions)
    ]

    if len(images) < (num_images_avg + num_images_test):
        continue

    # ---------- STEP 1: AGGREGATE (first 100 images) ----------
    avg_activations = []

    for img_path in images[:num_images_avg]:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)
            feat = get_activation(img)
            avg_activations.append(feat)
        except:
            continue

    avg_activations = np.stack(avg_activations)

    # 🔥 aggregate first
    avg_dist = avg_activations.mean(axis=0)

    # 🔥 normalize ONCE here
    avg_dist = avg_dist / (avg_dist.sum() + 1e-8)

    # ---------- STEP 2: KL for last 25 ----------
    kl_values = []

    for img_path in images[-num_images_test:]:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)
            feat = get_activation(img)

            # 🔥 normalize ONLY here for KL
            feat = feat / (feat.sum() + 1e-8)

            kl = entropy(feat + 1e-8, avg_dist + 1e-8)
            kl_values.append(kl)

        except:
            continue

    if len(kl_values) == 0:
        continue

    avg_kl = np.mean(kl_values)
    results[subfolder] = avg_kl

    print(f"{subfolder}: Avg KL = {avg_kl:.6f}")

# -------- FINAL OUTPUT --------
print("\n=== FINAL RESULTS (MobileNetV2 Improved) ===")
for k, v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{k}: {v:.6f}")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /Users/jaeeponde/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 19.0MB/s]


Cars: Avg KL = 0.285708
Cats: Avg KL = 0.316996
Dogs: Avg KL = 0.347401
Rangoli: Avg KL = 0.567002
memes: Avg KL = 0.374124
microscopy: Avg KL = 0.567915

=== FINAL RESULTS (MobileNetV2 Improved) ===
microscopy: 0.567915
Rangoli: 0.567002
memes: 0.374124
Dogs: 0.347401
Cats: 0.316996
Cars: 0.285708


In [17]:
# ================================
# VGG16 KL Divergence Pipeline
# ================================

import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy

# -------- CONFIG --------
base_path = "/Users/jaeeponde/Thesis/Data_100"
num_images_avg = 100
num_images_test = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

# -------- MODEL --------
model_full = models.vgg16(pretrained=True).to(device)
model_full.eval()

# penultimate = convolutional features + avgpool
model = nn.Sequential(
    model_full.features,
    model_full.avgpool
)

# -------- TRANSFORMS --------
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image(path):
    return transform(Image.open(path).convert("RGB"))

# -------- ACTIVATION FUNCTION --------
def get_activation(img_tensor):
    with torch.no_grad():
        x = model(img_tensor)  # (1, 512, 7, 7)
        feat = x.view(-1)      # 🔥 25088-dim

        # min-max normalize ONLY
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)

        return feat.cpu().numpy()

# -------- MAIN PIPELINE --------
results = {}

for subfolder in sorted(os.listdir(base_path)):
    subfolder_path = os.path.join(base_path, subfolder)

    if not os.path.isdir(subfolder_path):
        continue

    # Collect image paths
    images = [
        os.path.join(subfolder_path, f)
        for f in sorted(os.listdir(subfolder_path))
        if f.lower().endswith(image_extensions)
    ]

    # Ensure enough images
    if len(images) < (num_images_avg + num_images_test):
        continue

    # =========================
    # STEP 1: BUILD CLASS PROTOTYPE
    # =========================
    avg_activations = []

    for img_path in images[:num_images_avg]:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)
            feat = get_activation(img)
            avg_activations.append(feat)
        except Exception as e:
            print(f"Error (avg): {img_path} -> {e}")

    if len(avg_activations) == 0:
        continue

    avg_activations = np.stack(avg_activations)

    # Aggregate FIRST
    avg_dist = avg_activations.mean(axis=0)

    # Normalize ONCE
    avg_dist = avg_dist / (avg_dist.sum() + 1e-8)

    # =========================
    # STEP 2: KL DIVERGENCE
    # =========================
    kl_values = []

    for img_path in images[-num_images_test:]:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)
            feat = get_activation(img)

            # Normalize ONLY here
            feat = feat / (feat.sum() + 1e-8)

            kl = entropy(feat + 1e-8, avg_dist + 1e-8)
            kl_values.append(kl)

        except Exception as e:
            print(f"Error (KL): {img_path} -> {e}")

    if len(kl_values) == 0:
        continue

    avg_kl = np.mean(kl_values)
    results[subfolder] = avg_kl

    print(f"{subfolder}: Avg KL = {avg_kl:.6f}")

# =========================
# FINAL RESULTS
# =========================
print("\n=== FINAL RESULTS (VGG16 Improved) ===")

for k, v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{k}: {v:.6f}")

Cars: Avg KL = 1.872971
Cats: Avg KL = 1.900791
Dogs: Avg KL = 1.894323
Rangoli: Avg KL = 2.405987
memes: Avg KL = 2.173393
microscopy: Avg KL = 1.911239

=== FINAL RESULTS (VGG16 Improved) ===
Rangoli: 2.405987
memes: 2.173393
microscopy: 1.911239
Cats: 1.900791
Dogs: 1.894323
Cars: 1.872971


In [18]:
# ================================
# InceptionV3 KL Divergence Pipeline (CORRECT VERSION)
# ================================

import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy

# -------- CONFIG --------
base_path = "/Users/jaeeponde/Thesis/Data_100"
num_images_avg = 100
num_images_test = 100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

# -------- MODEL --------
class InceptionFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.inception_v3(pretrained=True)
        self.model.aux_logits = False

    def forward(self, x):
        # Copy of torchvision forward BUT stopping before fc
        x = self.model._transform_input(x)

        x = self.model.Conv2d_1a_3x3(x)
        x = self.model.Conv2d_2a_3x3(x)
        x = self.model.Conv2d_2b_3x3(x)
        x = self.model.maxpool1(x)

        x = self.model.Conv2d_3b_1x1(x)
        x = self.model.Conv2d_4a_3x3(x)
        x = self.model.maxpool2(x)

        x = self.model.Mixed_5b(x)
        x = self.model.Mixed_5c(x)
        x = self.model.Mixed_5d(x)

        x = self.model.Mixed_6a(x)
        x = self.model.Mixed_6b(x)
        x = self.model.Mixed_6c(x)
        x = self.model.Mixed_6d(x)
        x = self.model.Mixed_6e(x)

        x = self.model.Mixed_7a(x)
        x = self.model.Mixed_7b(x)
        x = self.model.Mixed_7c(x)

        x = self.model.avgpool(x)
        x = torch.flatten(x, 1)  # 🔥 2048-d features

        return x


model = InceptionFeatureExtractor().to(device)
model.eval()

# -------- TRANSFORMS --------
transform = transforms.Compose([
    transforms.Resize((299, 299)),  # required
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def load_image(path):
    return transform(Image.open(path).convert("RGB"))

# -------- ACTIVATION --------
def get_activation(img_tensor):
    with torch.no_grad():
        feat = model(img_tensor).view(-1)

        # min-max normalize ONLY
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)

        return feat.cpu().numpy()

# -------- MAIN --------
results = {}

for subfolder in sorted(os.listdir(base_path)):
    subfolder_path = os.path.join(base_path, subfolder)

    if not os.path.isdir(subfolder_path):
        continue

    images = [
        os.path.join(subfolder_path, f)
        for f in sorted(os.listdir(subfolder_path))
        if f.lower().endswith(image_extensions)
    ]

    if len(images) < (num_images_avg + num_images_test):
        continue

    # ---------- STEP 1: PROTOTYPE ----------
    avg_activations = []

    for img_path in images[:num_images_avg]:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)
            feat = get_activation(img)
            avg_activations.append(feat)
        except Exception as e:
            print(f"Error (avg): {img_path} -> {e}")

    if len(avg_activations) == 0:
        continue

    avg_activations = np.stack(avg_activations)

    avg_dist = avg_activations.mean(axis=0)
    avg_dist = avg_dist / (avg_dist.sum() + 1e-8)

    # ---------- STEP 2: KL ----------
    kl_values = []

    for img_path in images[-num_images_test:]:
        try:
            img = load_image(img_path).unsqueeze(0).to(device)
            feat = get_activation(img)

            feat = feat / (feat.sum() + 1e-8)

            kl = entropy(feat + 1e-8, avg_dist + 1e-8)
            kl_values.append(kl)

        except Exception as e:
            print(f"Error (KL): {img_path} -> {e}")

    if len(kl_values) == 0:
        continue

    avg_kl = np.mean(kl_values)
    results[subfolder] = avg_kl

    print(f"{subfolder}: Avg KL = {avg_kl:.6f}")

# -------- FINAL --------
print("\n=== FINAL RESULTS (InceptionV3 Correct) ===")

for k, v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{k}: {v:.6f}")

Cars: Avg KL = 0.277040
Cats: Avg KL = 0.254471
Dogs: Avg KL = 0.405114
Rangoli: Avg KL = 0.453183
memes: Avg KL = 0.325595
microscopy: Avg KL = 0.471053

=== FINAL RESULTS (InceptionV3 Correct) ===
microscopy: 0.471053
Rangoli: 0.453183
Dogs: 0.405114
memes: 0.325595
Cars: 0.277040
Cats: 0.254471
